# Détection de spam — Notebook révisé

Cette version conserve une organisation en **neuf parties** et ajoute les contrôles et les analyses demandés. Le modèle principal possède **45 couches cachées** (la sortie est une couche supplémentaire), utilise ReLU, l'initialisation He et **1 001 époques**, sans arrêt anticipé. Le réseau à trois couches est un contrôle expérimental.

**État des résultats :** les sorties de l'ancienne version ont été retirées. Les nouveaux entraînements complets ne sont pas présentés comme déjà réalisés. Exécuter les cellules dans l'ordre pour produire les tableaux et les figures de cette version. Les contrôles courts utilisés pour vérifier le code ne constituent pas des résultats scientifiques.

**Vérification de cette livraison :** structure du notebook validée avec `nbformat` et neuf parties exécutées sur une copie en mémoire à budget réduit (200 SMS synthétiques, un essai de sélection, deux époques par entraînement, une graine finale). Les contrôles incluent checkpoints, baselines, figures, statistiques appariées et exports. Aucun score de cette vérification n'est inclus dans les résultats du notebook.

Le vocabulaire et l'IDF sont appris sur l'apprentissage uniquement. Une validation séparée sert à la sélection et au choix d'un checkpoint diagnostique. Les deux ensembles de test restent réservés à l'évaluation finale. Un même réglage est utilisé pour les trois représentations afin de faciliter leur comparaison.

Le budget par défaut comprend 20 essais Optuna sur trois représentations et trois graines, puis deux finalistes sur deux graines supplémentaires : **192 entraînements de sélection**, auxquels s'ajoutent **50 entraînements finaux et contrôles**, tous à 1 001 époques. Les fichiers sont conservés dans `resultats_corriges/`. Ce protocole demande un temps de calcul conséquent ; son coût fait partie de la discussion finale.

Les résultats historiques cités dans les commentaires sont identifiés comme tels. Ils ne doivent pas être mélangés aux nouveaux résultats : le TF-IDF et la séparation apprentissage/validation ont changé.


## 1 — Installation et chargement

La cellule d'installation précède tout import d'Optuna. Le notebook utilise NumPy, pandas, Matplotlib, SciPy, scikit-learn, PyTorch et Optuna. Les versions effectivement utilisées sont affichées pour la reproductibilité.

Si `data/spam.csv` existe, il est chargé directement. Sinon, un téléchargement Kaggle est tenté ; son accessibilité sans authentification n'est pas garantie. En cas d'échec, télécharger le CSV depuis [la page du dataset](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset), puis le placer à cet emplacement ou modifier `DATA_FILE`. Aucun identifiant n'est inscrit dans le notebook. La présence d'un CSV local ne constitue pas un test du téléchargement.


In [ ]:
%pip install -q numpy pandas matplotlib scipy scikit-learn torch optuna


In [ ]:
from importlib.metadata import version
for package in ("numpy", "pandas", "matplotlib", "scipy", "scikit-learn", "torch", "optuna"):
    print(f"{package}: {version(package)}")


In [ ]:

from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile
from io import BytesIO

import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    display = print


# Configuration
SEED = 42

DATA_DIR = Path("data")
DATA_FILE = DATA_DIR / "spam.csv"  # Ou renseigner ici le chemin du CSV téléchargé manuellement.

DATA_URL = (
    "https://www.kaggle.com/api/v1/datasets/download/"
    "uciml/sms-spam-collection-dataset"
)

LABEL_MAPPING = {"ham": 0, "spam": 1}


# Télécharger les données directement depuis Kaggle
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_FILE.is_file():
    print("Téléchargement du dataset depuis Kaggle...")

    try:
        with urlopen(DATA_URL, timeout=30) as response:
            zip_content = response.read()

        with ZipFile(BytesIO(zip_content)) as archive:
            csv_content = archive.read("spam.csv")

        DATA_FILE.write_bytes(csv_content)

        print("Téléchargement terminé.")

    except Exception as error:
        raise RuntimeError(
            "Téléchargement Kaggle impossible. "
            "Vérifie ta connexion Internet ou télécharge manuellement "
            "le fichier spam.csv depuis Kaggle et place-le "
            "dans le dossier data."
        ) from error

else:
    print("Le fichier Kaggle spam.csv est déjà présent.")


# Charger le CSV Kaggle
# L'encodage latin-1 permet de lire les caractères de ce fichier.
df_raw = pd.read_csv(
    DATA_FILE,
    encoding="latin-1",
    dtype=str,
    keep_default_na=False
)

print("\nColonnes présentes dans le fichier Kaggle :")
print(df_raw.columns.tolist())


# Sélectionner les colonnes utiles
# v1 : catégorie ham/spam
# v2 : texte du message
required_columns = {"v1", "v2"}

if not required_columns.issubset(df_raw.columns):
    raise ValueError(
        "Le fichier doit contenir les colonnes 'v1' et 'v2'. "
        "Vérifie que tu utilises le bon dataset Kaggle."
    )

df = (
    df_raw[["v1", "v2"]]
    .copy()
    .rename(columns={
        "v1": "label_text",
        "v2": "message"
    })
)


# Vérifier les données
if df.empty:
    raise ValueError("Le dataset est vide.")

if df.isna().any().any():
    raise ValueError("Certaines valeurs utiles sont manquantes.")

df["label_text"] = df["label_text"].str.strip().str.lower()

if set(df["label_text"].unique()) != {"ham", "spam"}:
    raise ValueError(
        "Les catégories attendues sont exactement 'ham' et 'spam'."
    )

if df["message"].str.strip().eq("").any():
    raise ValueError("Le dataset contient des messages vides.")


# Convertir les étiquettes : ham = 0 et spam = 1
df["label"] = (
    df["label_text"]
    .map(LABEL_MAPPING)
    .astype("int64")
)


# Afficher les informations générales
print(f"\nNombre total de messages : {len(df)}")
print(f"Fichier utilisé : {DATA_FILE.resolve()}")

print("\nTypes des colonnes :")
print(df.dtypes)

print("\nAperçu des données :")
display(df.head())


# Calculer la répartition des catégories
class_counts = (
    df["label_text"]
    .value_counts()
    .reindex(["ham", "spam"], fill_value=0)
)

class_summary = pd.DataFrame({
    "Nombre": class_counts,
    "Pourcentage (%)": 100 * class_counts / len(df)
})

class_summary.index.name = "Classe"

print("\nRépartition initiale ham / spam :")
display(class_summary.round(2))


# Compter les messages répétés
# Ils seront traités avant le découpage entraînement / test.
n_duplicates = int(
    df.duplicated(subset=["message"]).sum()
)

print(
    "\nMessages répétés au-delà de la première occurrence : "
    f"{n_duplicates}"
)


# Afficher deux exemples de chaque catégorie
for category in ["ham", "spam"]:
    print(f"\nExemples de messages {category} :")

    subset = df.loc[
        df["label_text"] == category,
        ["label_text", "label", "message"]
    ]

    display(
        subset.sample(
            n=min(2, len(subset)),
            random_state=SEED
        )
    )


# Visualiser le déséquilibre des catégories
fig, ax = plt.subplots(figsize=(7, 4))

bars = ax.bar(
    class_counts.index,
    class_counts.values,
    color=["#37856B", "#CD5C5C"]
)

for bar, count in zip(bars, class_counts.values):
    percentage = 100 * count / len(df)

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        count,
        f"{count}\n({percentage:.1f} %)",
        ha="center",
        va="bottom"
    )

ax.set_title("Répartition initiale des messages — Dataset Kaggle")
ax.set_xlabel("Catégorie")
ax.set_ylabel("Nombre de messages")
ax.set_ylim(0, class_counts.max() * 1.2)

plt.tight_layout()
plt.show()



## 2 — Préparation, séparation et prévalence

Les doublons exacts sont supprimés avant le découpage ; les étiquettes contradictoires provoquent une erreur. La séparation stratifiée initiale réserve 20 % des messages à `test_pool`, qui conserve sa prévalence observée. Le test équilibré est obtenu ensuite par sous-échantillonnage de ce pool.

Les données de développement équilibrées sont séparées en apprentissage (80 %) et validation (20 %). Le vocabulaire, l'IDF et les éventuelles standardisations utilisent seulement l'apprentissage ; la validation sert à Optuna et au choix des checkpoints. Les effectifs affichés ici remplacent les 1 044 exemples d'apprentissage de l'ancien protocole : la validation réduit l'effectif disponible pour ajuster les poids.

Les deux tests partagent des messages : leurs résultats ne sont pas deux réplications indépendantes. Les graines d'entraînement font varier l'initialisation et l'ordre des mini-lots, en gardant **les mêmes partitions et les mêmes messages de test**. Elles ne mesurent pas l'incertitude due au choix du découpage.


In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

TEST_SIZE = 0.20
VALIDATION_SIZE = 0.20

conflicts = df.groupby('message')['label'].nunique()
if conflicts.gt(1).any():
    raise ValueError('Des messages identiques portent des labels contradictoires.')
df_clean = df.drop_duplicates('message').reset_index(drop=True).copy()
df_clean['message_id'] = np.arange(len(df_clean), dtype=np.int64)

# Le test_pool reste à sa prévalence observée et ne sert jamais à la sélection.
train_pool, test_pool = train_test_split(
    df_clean, test_size=TEST_SIZE, stratify=df_clean['label'], random_state=SEED
)
test_pool = test_pool.copy().reset_index(drop=True)

def balance_classes(data, seed):
    n = int(data['label'].value_counts().min())
    return (pd.concat([data.loc[data['label'].eq(c)].sample(n=n, random_state=seed)
                       for c in (0, 1)])
            .sample(frac=1, random_state=seed).reset_index(drop=True).copy())

df_train = balance_classes(train_pool, SEED)
test_df = balance_classes(test_pool, SEED + 1)
# Une validation fixe est réservée au sein des données d'apprentissage équilibrées.
train_df, val_df = train_test_split(
    df_train, test_size=VALIDATION_SIZE, stratify=df_train['label'], random_state=SEED + 2
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
df_test = test_df  # alias de lecture pour les anciennes notations du sujet

for left, right in ((train_df, val_df), (train_df, test_pool), (val_df, test_pool)):
    assert set(left['message_id']).isdisjoint(right['message_id'])
assert set(test_df['message_id']).issubset(test_pool['message_id'])

train_texts = train_df['message'].tolist()
val_texts = val_df['message'].tolist()
test_texts = test_df['message'].tolist()
test_pool_texts = test_pool['message'].tolist()
y_train = train_df['label'].to_numpy(dtype=np.float32)
y_val = val_df['label'].to_numpy(dtype=np.float32)
y_test = test_df['label'].to_numpy(dtype=np.float32)
y_test_pool = test_pool['label'].to_numpy(dtype=np.float32)

split_summary = pd.DataFrame([
    {'ensemble': name, 'n': len(part), 'ham': int(part['label'].eq(0).sum()),
     'spam': int(part['label'].eq(1).sum()), 'prevalence_spam': part['label'].mean()}
    for name, part in [('apprentissage', train_df), ('validation', val_df),
                       ('test_equilibre', test_df), ('test_pool', test_pool)]
]).set_index('ensemble')
display(split_summary)
print(f'{len(df) - len(df_clean)} doublons retirés ; aucun chevauchement entraînement/test.')
print('Le test équilibré est un sous-ensemble du test_pool : ces deux évaluations sont dépendantes.')


## 3 — Tokenisation et vocabulaire

Les règles fixes passent le texte en minuscules, remplacent les URL et extraient mots et symboles utiles. Elles ne sont pas apprises sur le test. Le vocabulaire classe les tokens par fréquence d'apprentissage, avec un ordre déterministe pour les ex æquo.

Les statistiques de couverture et de longueur décrivent seulement l'apprentissage. La sélection de la taille a lieu en **partie 5**, avec les autres paramètres. L'ancienne fonction `search_vocab_size` et la configuration `VOCAB_OPTIONS` sont supprimées, ainsi que les commentaires qui annonçaient leur utilisation.

« Binaire » désigne ici la présence d'un token dans un vecteur par message (multi-hot), plutôt qu'un unique one-hot par message.


In [ ]:
import re
from collections import Counter

REPRESENTATIONS = ('binaire', 'bow', 'tfidf')
TOKENIZATION_VERSION = 'url-lower-unicode-v2'
URL_PATTERN = re.compile(r'(?:https?://|www\.)\S+', flags=re.IGNORECASE)
TOKEN_PATTERN = re.compile(r"<url>|[^\W_]+(?:'[^\W_]+)*|[£$€!?]", flags=re.UNICODE)

def tokenize(text):
    return TOKEN_PATTERN.findall(URL_PATTERN.sub(' <url> ', text.lower().replace('’', "'")))

def build_vocabulary(messages_tokens, max_size=None):
    counts = Counter(token for tokens in messages_tokens for token in tokens)
    if not counts:
        raise ValueError('Aucun token dans cet apprentissage.')
    ranked = sorted(counts.items(), key=lambda item: (-item[1], item[0]))
    if max_size is not None:
        ranked = ranked[:int(max_size)]
    words = [word for word, _ in ranked]
    return {'vocabulaire': words, 'token_to_index': {w: i for i, w in enumerate(words)},
            'V': len(words)}

train_tokens = [tokenize(t) for t in train_texts]
val_tokens = [tokenize(t) for t in val_texts]
test_tokens = [tokenize(t) for t in test_texts]
test_pool_tokens = [tokenize(t) for t in test_pool_texts]
token_counts = Counter(t for doc in train_tokens for t in doc)
coverage_table = pd.DataFrame([
    {'limite': str(limit), 'dimension': (v := build_vocabulary(train_tokens, limit))['V'],
     'couverture_occurrences': sum(token_counts[w] for w in v['vocabulaire']) / sum(token_counts.values())}
    for limit in (500, 1000, 2000, 3000, None)
])
display(coverage_table)
display(pd.Series([len(t) for t in train_tokens], name='tokens_par_message').describe())
print('Statistiques descriptives uniquement. La sélection du vocabulaire a lieu en partie 5.')


## 4 — Binaire, BoW et définition du TF-IDF

Le binaire indique la présence de chaque token et BoW compte ses occurrences. Pour la nouvelle expérience principale :

\[
\mathrm{TF}(t,d)=\frac{n(t,d)}{\sum_u n(u,d)},\qquad
\mathrm{IDF}(t)=\ln\frac{N}{\mathrm{DF}(t)},\qquad
x_{t,d}=\mathrm{TF}(t,d)\,\mathrm{IDF}(t).
\]

Le dénominateur du TF comprend **tous les tokens du document**, y compris ceux hors vocabulaire. Un document sans token produit un vecteur nul. L'IDF non lissé est ajusté sur l'apprentissage exclusivement ; tous les tokens retenus y sont présents.

La fréquence relative et la normalisation L2 sont deux opérations distinctes. La représentation principale utilise le TF relatif **sans normalisation L2 supplémentaire**. La fonction expose aussi `tf_mode='raw'` et `l2=True` pour des expériences complémentaires explicitement identifiées ; ces options ne sont pas employées silencieusement.

L'ancienne version utilisait le comptage brut, autre convention courante du TF-IDF. Ses BCE initiales et ses vitesses de convergence ne sont donc pas directement comparables aux nouvelles. Une différence de vitesse ne peut pas être attribuée automatiquement à ce choix : il faudrait une ablation conservant tous les autres réglages. Les nouvelles matrices et le code du modèle entrent dans les empreintes des expériences ; les anciens checkpoints ne sont pas rechargés.


In [ ]:
VECTORIZATION_VERSION = 'tf-relative-all-tokens-idf-log-N-df-no-l2-v2'

def vectorize_bow(messages_tokens, token_to_index):
    X = np.zeros((len(messages_tokens), len(token_to_index)), dtype=np.float32)
    for row, tokens in enumerate(messages_tokens):
        for token, count in Counter(tokens).items():
            column = token_to_index.get(token)
            if column is not None:
                X[row, column] = count
    return X

def vectorize_binary(messages_tokens, token_to_index):
    return (vectorize_bow(messages_tokens, token_to_index) > 0).astype(np.float32)

def fit_idf(tokens_train, token_to_index):
    counts = np.zeros(len(token_to_index), dtype=np.int64)
    for tokens in tokens_train:
        for token in set(tokens):
            column = token_to_index.get(token)
            if column is not None:
                counts[column] += 1
    if not len(tokens_train) or not len(token_to_index) or np.any(counts == 0):
        raise ValueError('Apprendre le vocabulaire et les IDF sur le même apprentissage non vide.')
    return np.log(len(tokens_train) / counts).astype(np.float32)

def vectorize_tfidf(messages_tokens, token_to_index, idf, tf_mode='relative', l2=False):
    idf = np.asarray(idf, dtype=np.float32)
    if idf.shape != (len(token_to_index),):
        raise ValueError('Dimension IDF incorrecte.')
    X = vectorize_bow(messages_tokens, token_to_index)
    if tf_mode == 'relative':
        # Dénominateur = TOUS les tokens du document, y compris hors vocabulaire.
        lengths = np.asarray([max(1, len(t)) for t in messages_tokens], dtype=np.float32)
        X /= lengths[:, None]
    elif tf_mode != 'raw':
        raise ValueError('tf_mode doit être relative ou raw.')
    X *= idf
    if l2:
        norms = np.linalg.norm(X, axis=1, keepdims=True)
        X = np.divide(X, norms, out=np.zeros_like(X), where=norms > 0)
    return X

def prepare_vectors(tokens_train, tokens_other, token_to_index, representation):
    idf = None
    if representation == 'tfidf':
        idf = fit_idf(tokens_train, token_to_index)
        X_train = vectorize_tfidf(tokens_train, token_to_index, idf)
        X_other = vectorize_tfidf(tokens_other, token_to_index, idf)
    else:
        fn = {'binaire': vectorize_binary, 'bow': vectorize_bow}[representation]
        X_train, X_other = fn(tokens_train, token_to_index), fn(tokens_other, token_to_index)
    return {'X_train': X_train, 'X_other': X_other, 'idf': idf,
            'V': len(token_to_index), 'representation': representation}

demo_train = [tokenize('Free free call'), tokenize('Meeting call')]
demo_other = [tokenize('Free mystery'), tokenize('...')]
demo_vocab = build_vocabulary(demo_train)
for rep in REPRESENTATIONS:
    demo = prepare_vectors(demo_train, demo_other, demo_vocab['token_to_index'], rep)
    print(rep)
    display(pd.DataFrame(demo['X_other'], columns=demo_vocab['vocabulaire']))
print('TF-IDF principal : TF relatif, IDF = ln(N/DF), sans normalisation L2.')


## 5 — Sélection et incertitude de la recherche

Optuna explore **20 configurations**, avec trois graines par configuration et les trois représentations. L'objectif est la moyenne du F1 de validation, avec un seuil fixé à 0,5. Les deux finalistes sont évalués sur deux graines supplémentaires ; les tableaux exposent les scores et leur dispersion. La configuration retenue est commune aux trois représentations.

Les choix sont bornés : vocabulaire de 500 à 3 000 tokens (dans la limite des tokens présents), largeur parmi 64, 128, 192 et 256, taux d'apprentissage logarithmique de $10^{-4}$ à $3\times10^{-3}$, mini-lots parmi 32, 64, 128 et 256. La profondeur reste à 45 couches cachées et la durée à 1 001 époques. Ces bornes limitent le coût et évitent les largeurs 1 ou 2 trop restrictives ; elles ne garantissent pas l'absence d'entraînements défaillants.

Il s'agit de la **meilleure configuration trouvée dans cet espace et ce budget**, sans garantie d'optimum global. La validation est un seul découpage, réutilisé pour la sélection : un faible écart entre finalistes, par exemple 0,011 dans les anciens essais, ne suffit pas à établir un classement fiable. Les nouvelles répétitions réduisent une partie de l'incertitude liée aux graines, sans annuler l'effet de sélection.

L'étude est enregistrée dans un répertoire dont l'empreinte inclut données, transformations réelles, code, versions et réglages. Une étude correspondante peut reprendre les essais terminés ; elle ne justifie jamais de réutiliser une étude issue d'une ancienne vectorisation. Le test n'intervient ni dans l'objectif ni dans la confirmation des finalistes.


In [ ]:
import gc
import hashlib
import json
import inspect
import marshal
import random
from pathlib import Path
from time import perf_counter

import optuna
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Un petit nombre de threads évite de pénaliser les petites matrices sur CPU.
if DEVICE.type == 'cpu':
    torch.set_num_threads(min(4, torch.get_num_threads()))
MAIN_HIDDEN_LAYERS = 45
MAIN_EPOCHS = 1001
SEARCH_SEEDS = (42, 43, 44)
CONFIRMATION_SEEDS = (45, 46)
FINAL_SEEDS = (42, 43, 44, 45, 46)
N_TRIALS = 20
N_FINALISTS = 2
SEARCH_SPACE = {
    'vocab_size': [500, 1000, 2000, 3000],
    'hidden_dim': [64, 128, 192, 256],
    'learning_rate': [1e-4, 3e-3],
    'batch_size': [32, 64, 128, 256],
    'hidden_layers_fixed': MAIN_HIDDEN_LAYERS,
    'epochs_fixed': MAIN_EPOCHS,
}

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class SpamDNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, hidden_layers=45, init='he'):
        super().__init__()
        if min(input_dim, hidden_dim, hidden_layers) < 1:
            raise ValueError('Les dimensions doivent être positives.')
        if init not in ('he', 'xavier', 'pytorch', 'normal_001'):
            raise ValueError('Initialisation inconnue.')
        self.hidden_layers = int(hidden_layers)
        self.init_name = init
        self.hidden = nn.Sequential(*[
            layer for i in range(hidden_layers)
            for layer in (nn.Linear(input_dim if i == 0 else hidden_dim, hidden_dim), nn.ReLU())
        ])
        self.output = nn.Linear(hidden_dim, 1)
        for layer in self.modules():
            if isinstance(layer, nn.Linear):
                if init == 'he':
                    nn.init.kaiming_normal_(layer.weight, mode='fan_in', nonlinearity='relu')
                elif init == 'xavier':
                    nn.init.xavier_normal_(layer.weight, gain=1.0)
                elif init == 'normal_001':
                    nn.init.normal_(layer.weight, mean=0.0, std=0.01)
                # pytorch conserve les poids du constructeur Linear : U(-1/sqrt(fan_in), +...).
                nn.init.zeros_(layer.bias)  # contrôle commun à toutes les cellules du plan

    def forward(self, x):
        return self.output(self.hidden(x)).squeeze(-1)

def score_candidate(config, seeds, representations=REPRESENTATIONS):
    """Sélection sur validation uniquement ; aucun accès aux deux tests."""
    vocab = build_vocabulary(train_tokens, config['vocab_size'])
    mapping = vocab['token_to_index']
    rows = []
    for rep in representations:
        vec = prepare_vectors(train_tokens, val_tokens, mapping, rep)
        X_fit = torch.as_tensor(vec['X_train'], dtype=torch.float32, device=DEVICE)
        y_fit = torch.as_tensor(y_train, dtype=torch.float32, device=DEVICE)
        X_v = torch.as_tensor(vec['X_other'], dtype=torch.float32, device=DEVICE)
        for seed in seeds:
            set_seed(seed)
            model = SpamDNN(X_fit.shape[1], config['hidden_dim'], MAIN_HIDDEN_LAYERS, 'he').to(DEVICE)
            optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])
            criterion = nn.BCEWithLogitsLoss()
            generator = torch.Generator().manual_seed(seed)
            failed = False
            for epoch in range(config['epochs']):
                model.train()
                permutation = torch.randperm(len(y_fit), generator=generator)
                for start in range(0, len(y_fit), config['batch_size']):
                    idx = permutation[start:start + config['batch_size']].to(DEVICE)
                    optimizer.zero_grad(set_to_none=True)
                    loss = criterion(model(X_fit[idx]), y_fit[idx])
                    if not torch.isfinite(loss):
                        failed = True
                        break
                    loss.backward()
                    optimizer.step()
                if failed:
                    break
            model.eval()
            with torch.no_grad():
                logits = model(X_v).cpu().numpy()
            failed = failed or not np.isfinite(logits).all()
            rows.append({'representation': rep, 'seed': int(seed),
                         'f1': 0.0 if failed else float(f1_score(y_val, logits >= 0, zero_division=0)),
                         'failed': bool(failed)})
            del model, optimizer
            gc.collect()
            if DEVICE.type == 'cuda':
                torch.cuda.empty_cache()
    return rows

def run_search(n_trials=N_TRIALS):
    # Empreinte des transformations réelles, au-delà de leur nom de version.
    matrix_hashes = {}
    for limit in SEARCH_SPACE['vocab_size']:
        mapping = build_vocabulary(train_tokens, limit)['token_to_index']
        matrix_hashes[str(limit)] = {}
        for rep in REPRESENTATIONS:
            vectors = prepare_vectors(train_tokens, val_tokens, mapping, rep)
            matrix_hashes[str(limit)][rep] = {
                key: hashlib.sha256(np.ascontiguousarray(vectors[key]).tobytes()).hexdigest()
                for key in ('X_train', 'X_other')
            }
    code_hashes = {}
    for name, function in [('score_candidate', score_candidate), ('set_seed', set_seed),
                           ('model_init', SpamDNN.__init__), ('model_forward', SpamDNN.forward)]:
        try:
            source_bytes = inspect.getsource(function).encode()
        except (OSError, TypeError):
            source_bytes = marshal.dumps(function.__code__)
        code_hashes[name] = hashlib.sha256(source_bytes).hexdigest()
    protocol = {
        'version': 'selection-v3-common-representations-relative-tf',
        'tokenization': TOKENIZATION_VERSION, 'vectorization': VECTORIZATION_VERSION,
        'train': train_df[['message_id', 'message', 'label']].to_dict('records'),
        'validation': val_df[['message_id', 'message', 'label']].to_dict('records'),
        'space': SEARCH_SPACE, 'search_seeds': SEARCH_SEEDS,
        'confirmation_seeds': CONFIRMATION_SEEDS, 'torch': str(torch.__version__),
        'optuna': optuna.__version__, 'device': str(DEVICE),
        'biases': 'zero', 'optimizer': 'Adam-defaults-except-lr',
        'matrix_hashes': matrix_hashes, 'code_hashes': code_hashes,
    }
    digest = hashlib.sha256(json.dumps(protocol, sort_keys=True, ensure_ascii=False).encode()).hexdigest()[:16]
    folder = Path('resultats_corriges') / ('selection_' + digest)
    folder.mkdir(parents=True, exist_ok=True)
    (folder / 'protocole.json').write_text(json.dumps(protocol, indent=2, ensure_ascii=False), encoding='utf-8')
    study = optuna.create_study(
        direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED),
        storage='sqlite:///' + str((folder / 'study.sqlite3').resolve()),
        study_name='F1-validation-moyen-3-representations-3-graines', load_if_exists=True
    )

    def objective(trial):
        config = {
            'vocab_size': trial.suggest_categorical('vocab_size', SEARCH_SPACE['vocab_size']),
            'hidden_dim': trial.suggest_categorical('hidden_dim', SEARCH_SPACE['hidden_dim']),
            'learning_rate': trial.suggest_float('learning_rate', *SEARCH_SPACE['learning_rate'], log=True),
            'batch_size': trial.suggest_categorical('batch_size', SEARCH_SPACE['batch_size']),
            'epochs': MAIN_EPOCHS,
        }
        rows = score_candidate(config, SEARCH_SEEDS)
        trial.set_user_attr('validation_details', rows)
        return float(np.mean([row['f1'] for row in rows]))

    completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    remaining = max(0, int(n_trials) - len(completed))
    if remaining:
        study.optimize(objective, n_trials=remaining, gc_after_trial=True)
    completed = sorted([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE],
                       key=lambda t: (-t.value, t.number))
    if not completed:
        raise RuntimeError('Aucun essai Optuna terminé.')
    candidates = []
    details = []
    for trial in completed[:N_FINALISTS]:
        config = {**trial.params, 'epochs': MAIN_EPOCHS}
        # Confirmation sur deux graines supplémentaires, toujours sans consulter le test.
        extra = score_candidate(config, CONFIRMATION_SEEDS)
        all_rows = trial.user_attrs['validation_details'] + extra
        seed_means = pd.DataFrame(all_rows).groupby('seed')['f1'].mean()
        candidates.append({'trial': trial.number, 'mean_f1': seed_means.mean(),
                           'std_seed_f1': seed_means.std(ddof=1), 'config': config})
        details.extend([{'trial': trial.number, **r} for r in all_rows])
    ranked = sorted(candidates, key=lambda c: (-c['mean_f1'], c['trial']))
    best = dict(ranked[0]['config'])
    study.trials_dataframe().to_csv(folder / 'essais.csv', index=False)
    pd.DataFrame(details).to_csv(folder / 'finalistes_par_graine_representation.csv', index=False)
    (folder / 'meilleure_configuration_trouvee.json').write_text(
        json.dumps(best, indent=2), encoding='utf-8')
    table = pd.DataFrame([{k: v for k, v in c.items() if k != 'config'} for c in ranked])
    display(table)
    print('Meilleure configuration trouvée dans ce budget :', best)
    print('Une faible différence de validation ne suffit pas à établir une supériorité.')
    return best, study, table, folder

best_config, optuna_study, selection_summary, SEARCH_DIR = run_search()
# Configuration COMMUNE : seule la représentation varie dans la comparaison principale.
best_configs = {rep: dict(best_config) for rep in REPRESENTATIONS}
display(pd.DataFrame(best_configs).T)


## 6 — Entraînement, répétitions et contrôles

Les trois représentations sont entraînées avec **la même configuration retenue en validation**, 45 couches cachées, une initialisation de He, plus de 1 000 époques et cinq graines. Les données et les permutations des mini-batchs sont identiques entre conditions pour une graine donnée. Les métriques ne sont pas regroupées comme si cinq prédictions d'un même SMS étaient cinq observations indépendantes.

L'ablation utilise le TF-IDF fixé avant l'évaluation : **2 profondeurs (3 ou 45 couches cachées) × 4 initialisations (He, Xavier, poids par défaut de PyTorch, normale d'écart-type 0,01) × 5 graines**. Tous les biais sont initialisés à zéro puis appris normalement, y compris dans le contrôle « PyTorch », qui ne reprend donc que son initialisation par défaut des poids. Les recettes d’initialisation s’appliquent à toutes les matrices linéaires, sortie comprise : He normale (fan-in, ReLU), Xavier normale (gain 1), poids de `nn.Linear` laissés par défaut ou normale d’écart-type 0,01. Largeur, vocabulaire, TF-IDF, optimiseur Adam, taux d'apprentissage, mini-batchs et durée restent identiques. La sortie ajoute une couche linéaire à ces profondeurs. Le contrôle He/45 est le modèle principal déjà entraîné ; il n'est pas recalculé. La profondeur change aussi le nombre de paramètres : cette comparaison mesure l'effet global de la profondeur à largeur fixée, sans égaliser la capacité.

Un **même mini-batch sonde fixe** mesure le gradient de chaque matrice de poids : norme de Frobenius et RMS = norme / racine du nombre de poids. Les mesures `epoch=0` et `epoch=1` décrivent toutes deux l'état **avant le premier pas d'optimisation** ; les autres sondes précèdent l'époque indiquée. Elles ne mettent pas à jour le modèle. La perte de cette sonde est conservée pour ne pas confondre une petite norme due à une perte presque nulle avec une propagation défaillante. La norme est calculée en float64 pour éviter un sous-flux lors de sa réduction ; la rétropropagation elle-même reste en float32 et peut produire des zéros numériques dans les contrôles les plus profonds.

Chaque entraînement va au terme imposé. Le checkpoint final reste le résultat principal. Un second checkpoint minimise la BCE de validation : il sert à étudier l'effet de la durée sur la généralisation, sans sélectionner un modèle à partir du test. Cette comparaison est descriptive et ne démontre pas que les 1 000 époques expliquent à elles seules les différences de calibration.

Deux références sont ajoutées : régression logistique sur **le même TF-IDF**, et régression logistique sur des **features manuelles** (longueur, nombre de mots, proportions de majuscules et de chiffres, liens, signes monétaires, points d'exclamation et d'interrogation). Le scaler de cette dernière est ajusté uniquement sur l'entraînement. Les deux baselines testent la même grille `C ∈ {0,01 ; 0,1 ; 1 ; 10 ; 100 ; 1 000}` et retiennent le meilleur F1 de validation ; une égalité garde le plus petit C. Les choix et scores de validation sont sauvegardés, sans consultation du test. Le solveur `lbfgs` est déterministe sur ces données : ses répétitions vérifient le protocole mais ne représentent pas cinq apprentissages indépendants. Leurs performances doivent être mesurées avant toute conclusion sur leur proximité avec le DNN.

Les poids, vocabulaires, historiques, gradients et prédictions sont sauvegardés dans un répertoire identifié par une empreinte des données, des matrices encodées et du protocole. **Aucun ancien checkpoint n'est chargé automatiquement** : la modification du TF-IDF invalide les expériences antérieures. Les temps rapportés mesurent la boucle d'apprentissage (sondes et validation incluses pour les DNN) et tous les ajustements de la grille C pour les baselines, pas un benchmark complet de bout en bout. Le protocole complet représente 50 entraînements DNN (15 principaux + 35 contrôles supplémentaires), ce qui peut nécessiter plusieurs heures selon le matériel ; le mode de test rapide ne produit aucun résultat scientifique.

Une perte non finie interrompt explicitement le calcul avec le nom de la condition et l’époque. Cet échec doit être rapporté avant toute comparaison ; il ne faut pas supprimer la condition pour poursuivre les statistiques.


In [ ]:
# PARTIE 6 — Répétitions, contrôles croisés et baselines
import inspect
import hashlib
import json
import math
import marshal
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

FINAL_SEEDS = tuple(globals().get('FINAL_SEEDS', (42, 43, 44, 45, 46)))
ABLATION_REPRESENTATION = 'tfidf'
ABLATION_DEPTHS = (3, 45)  # Nombre de couches cachées ; sortie en supplément.
ABLATION_INITIALIZATIONS = ('he', 'xavier', 'pytorch', 'normal_001')
BASELINE_C_GRID = (0.01, 0.1, 1., 10., 100., 1000.)
RESULTS_DIR = Path('resultats_corriges')


def _message_ids(frame):
    """Identifiant stable : mêmes SMS = mêmes unités statistiques entre runs."""
    return np.array([hashlib.sha256(str(s).encode('utf-8')).hexdigest()
                     for s in frame['message']], dtype=str)


def _experiment_fingerprint(configs, frames, smoke, matrix_bundles):
    # Toute modification du corpus, découpage, vocabulaire, TF-IDF ou protocole
    # produit un nouveau répertoire. Aucun checkpoint ancien n'est rechargé.
    sources = {}
    for name in ('tokenize', 'build_vocabulary', 'prepare_vectors',
                 'vectorize_tfidf', 'fit_idf', 'SpamDNN', 'train_controlled_run',
                 '_gradient_probe', 'manual_features', 'run_experiments'):
        obj = globals().get(name)
        try:
            sources[name] = inspect.getsource(obj)
        except (OSError, TypeError):
            # Le source des cellules est parfois indisponible dans Jupyter.
            code = getattr(obj, '__code__', None)
            sources[name] = marshal.dumps(code).hex() if code else repr(obj)
    payload = {'protocol': 'relative-tf-full-token-denominator-no-l2-baseline-grid-v4',
               'baseline_C_grid': BASELINE_C_GRID, 'torch_version': torch.__version__,
               'configs': configs, 'seeds': FINAL_SEEDS,
               'biases': 'zero', 'depths': ABLATION_DEPTHS,
               'initializations': ABLATION_INITIALIZATIONS, 'smoke': smoke,
               'sources': sources,
               'matrix_hashes': {rep: {split: hashlib.sha256(
                   np.ascontiguousarray(matrix).tobytes()).hexdigest()
                   for split, matrix in bundle.items()}
                   for rep, bundle in matrix_bundles.items()},
               'frames': {k: list(zip(_message_ids(v).tolist(),
                                      v['label'].astype(int).tolist()))
                          for k, v in frames.items()}}
    return hashlib.sha256(json.dumps(payload, sort_keys=True,
                                    default=str).encode()).hexdigest()[:20]


def _vector_bundle(rep, config, frames):
    vocabulary = build_vocabulary(train_tokens,
                                  max_size=int(config['vocab_size']))
    mapping = vocabulary['token_to_index']
    train_val = prepare_vectors(train_tokens, val_tokens, mapping, rep)
    matrices = {'train': np.asarray(train_val['X_train'], dtype=np.float32),
                'val': np.asarray(train_val['X_other'], dtype=np.float32)}
    # IDF et vocabulaire restent ajustés exclusivement sur l'entraînement.
    for split, tokens in [('test_equilibre', test_tokens),
                          ('test_pool', test_pool_tokens)]:
        matrices[split] = np.asarray(
            prepare_vectors(train_tokens, tokens, mapping, rep)['X_other'],
            dtype=np.float32)
    return matrices, vocabulary


def _state_copy(model):
    return {name: value.detach().cpu().clone()
            for name, value in model.state_dict().items()}


def _batched_logits(model, matrix, batch_size=512):
    model.eval()
    device = next(model.parameters()).device
    chunks = []
    with torch.no_grad():
        for start in range(0, len(matrix), batch_size):
            chunks.append(model(torch.as_tensor(matrix[start:start + batch_size],
                                                device=device)).cpu().numpy())
    return np.concatenate(chunks).reshape(-1)


def _bce_from_logits(y, logits):
    return float(np.mean(np.logaddexp(0., np.asarray(logits, dtype=np.float64))
                         - np.asarray(y, dtype=np.float64) * logits))


def _gradient_probe(model, x_probe, y_probe, metadata, epoch):
    """Gradient moyen sur un même mini-batch fixé, sans pas d'optimisation."""
    model.train()
    model.zero_grad(set_to_none=True)
    probe_loss = nn.BCEWithLogitsLoss()(model(x_probe), y_probe)
    probe_loss.backward()
    linears = [(name, layer) for name, layer in model.named_modules()
               if isinstance(layer, nn.Linear)]
    rows = []
    for index, (name, layer) in enumerate(linears):
        gradient = layer.weight.grad.detach()
        norm = float(torch.linalg.vector_norm(gradient.to(dtype=torch.float64)).cpu())
        rows.append({**metadata, 'epoch': int(epoch), 'layer_index': index,
                     'layer_name': name, 'is_output': index == len(linears) - 1,
                     'fro': norm, 'rms': norm / math.sqrt(gradient.numel()),
                     'n_weights': gradient.numel(),
                     'probe_loss': float(probe_loss.detach().cpu())})
    model.zero_grad(set_to_none=True)
    return rows


def train_controlled_run(matrices, labels, config, rep, seed, depth, init,
                         run_dir):
    """Même protocole pour tous les contrôles ; aucun arrêt anticipé."""
    set_seed(int(seed))
    device = globals().get('DEVICE', torch.device('cpu'))
    model = SpamDNN(input_dim=matrices['train'].shape[1],
                    hidden_dim=int(config['hidden_dim']),
                    hidden_layers=int(depth), init=init).to(device)
    optimizer = torch.optim.Adam(model.parameters(),
                                 lr=float(config['learning_rate']))
    criterion = nn.BCEWithLogitsLoss()
    x_train = torch.as_tensor(matrices['train'], device=device)
    y_train_tensor = torch.as_tensor(labels['train'], dtype=torch.float32,
                                     device=device)
    epochs = int(config['epochs'])
    batch_size = int(config['batch_size'])
    run_id = f'{rep}_d{depth}_{init}_s{seed}'
    metadata = dict(run_id=run_id, representation=rep, seed=int(seed),
                    hidden_layers=int(depth), init=init)
    probe_size = min(batch_size, len(x_train))
    # Même sous-ensemble fixe dans tous les contrôles et toutes les graines.
    x_probe, y_probe = x_train[:probe_size], y_train_tensor[:probe_size]
    gradient_rows = _gradient_probe(model, x_probe, y_probe, metadata, epoch=0)
    initial_hidden = [r for r in gradient_rows if not r['is_output']]
    first, last = initial_hidden[0], initial_hidden[-1]
    gradient_ratio = first['fro'] / last['fro'] if last['fro'] > 0 else np.nan
    rms_ratio = first['rms'] / last['rms'] if last['rms'] > 0 else np.nan
    # Le générateur du sampler ne dépend jamais de l'initialisation du modèle.
    sampler_rng = torch.Generator(device='cpu').manual_seed(int(seed))
    history, best_val, best_epoch, best_state = [], float('inf'), None, None
    probe_epochs = {1, 2, 5, 10, 100, epochs}
    started = perf_counter()
    for epoch in range(1, epochs + 1):
        # epoch=1 : état avant le premier pas ; la lecture n'est pas confondue
        # avec une perte devenue presque nulle après convergence.
        if epoch in probe_epochs:
            gradient_rows.extend(_gradient_probe(model, x_probe, y_probe,
                                                 metadata, epoch))
        model.train()
        permutation = torch.randperm(len(x_train), generator=sampler_rng)
        for start in range(0, len(x_train), batch_size):
            indices = permutation[start:start + batch_size].to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(x_train[indices]), y_train_tensor[indices])
            if not bool(torch.isfinite(loss)):
                raise FloatingPointError(f'Perte non finie : {run_id}, époque {epoch}')
            loss.backward()
            optimizer.step()
        # BCE à un même checkpoint, et non moyenne des pertes avant chaque pas.
        train_bce = _bce_from_logits(labels['train'],
                                     _batched_logits(model, matrices['train']))
        val_bce = _bce_from_logits(labels['val'],
                                   _batched_logits(model, matrices['val']))
        if not np.isfinite(train_bce) or not np.isfinite(val_bce):
            raise FloatingPointError(f'BCE non finie : {run_id}, époque {epoch}')
        history.append({'epoch': epoch, 'train_bce': train_bce,
                        'val_bce': val_bce})
        if val_bce < best_val:
            best_val, best_epoch, best_state = val_bce, epoch, _state_copy(model)
    fit_seconds = perf_counter() - started
    final_state = _state_copy(model)
    run_dir.mkdir(parents=True, exist_ok=True)
    torch.save(final_state, run_dir / 'final.pt')
    torch.save(best_state, run_dir / 'best_val.pt')
    pd.DataFrame(history).to_csv(run_dir / 'history.csv', index=False)
    pd.DataFrame(gradient_rows).to_csv(run_dir / 'gradients.csv', index=False)
    info = {**metadata,
            'n_parameters': sum(p.numel() for p in model.parameters()),
            'fit_seconds': fit_seconds, 'epochs': epochs,
            'final_train_bce': history[-1]['train_bce'],
            'final_val_bce': history[-1]['val_bce'],
            'best_val_bce': best_val, 'best_val_epoch': best_epoch,
            'initial_gradient_ratio': gradient_ratio,
            'initial_rms_ratio': rms_ratio,
            'parameters_per_train_message':
                sum(p.numel() for p in model.parameters()) / len(x_train)}
    (run_dir / 'metadata.json').write_text(json.dumps(info, indent=2), encoding='utf-8')
    return model, final_state, best_state, history, gradient_rows, info


def manual_features(messages):
    """Features fixées avant tout examen du test ; texte original non nettoyé."""
    import re
    rows = []
    for text in messages:
        text = str(text)
        n = max(len(text), 1)
        letters = sum(c.isalpha() for c in text)
        rows.append([len(text), len(text.split()),
                     sum(c.isupper() for c in text) / max(letters, 1),
                     sum(c.isdigit() for c in text) / n,
                     len(re.findall(r'https?://|www\.', text, flags=re.I)),
                     text.count('£'), text.count('$'), text.count('€'),
                     text.count('!'), text.count('?')])
    return np.asarray(rows, dtype=np.float64)


def run_experiments(smoke=False):
    frames = {'train': train_df, 'val': val_df,
              'test_equilibre': test_df, 'test_pool': test_pool}
    labels = {name: frame['label'].to_numpy(dtype=np.float32)
              for name, frame in frames.items()}
    configs = {rep: dict(config) for rep, config in best_configs.items()}
    seeds = FINAL_SEEDS
    if not smoke:
        if len(seeds) < 5:
            raise ValueError('Le protocole final exige au moins cinq graines.')
        if any(int(c['epochs']) < 1001 for c in configs.values()):
            raise ValueError('Les modèles principaux gardent plus de 1000 époques.')
    else:
        # Seulement pour valider le pipeline : ne pas interpréter ces scores.
        seeds = seeds[:1]
        for c in configs.values():
            c.update(epochs=2, hidden_dim=8)
    matrices_by_rep, vocabularies = {}, {}
    for rep in ('binaire', 'bow', 'tfidf'):
        matrices_by_rep[rep], vocabularies[rep] = _vector_bundle(
            rep, configs[rep], frames)
    fingerprint = _experiment_fingerprint(configs, frames, smoke, matrices_by_rep)
    root = RESULTS_DIR / ('SMOKE_' + fingerprint if smoke else fingerprint)
    root.mkdir(parents=True, exist_ok=True)
    (root / 'protocole.json').write_text(json.dumps({
        'fingerprint': fingerprint, 'smoke_only': bool(smoke), 'configs': configs,
        'seeds': list(seeds), 'hidden_layers_main': 45,
        'ablation_representation': ABLATION_REPRESENTATION,
        'ablation_depths': list(ABLATION_DEPTHS),
        'ablation_initializations': list(ABLATION_INITIALIZATIONS),
        'biases_initially': 'zero', 'optimizer': 'Adam',
        'tokenization': globals().get('TOKENIZATION_VERSION'),
        'vectorization': globals().get('VECTORIZATION_VERSION'),
        'torch': str(torch.__version__), 'numpy': np.__version__,
        'device': str(globals().get('DEVICE', 'cpu')),
        'split_sizes': {key: len(frame) for key, frame in frames.items()},
        'baseline_C_grid': [0.01, 0.1, 1, 10, 100, 1000],
    }, indent=2, ensure_ascii=False), encoding='utf-8')
    histories, gradients, infos, records, artifacts = {}, [], [], [], {}
    for rep in ('binaire', 'bow', 'tfidf'):
        matrices, vocabulary = matrices_by_rep[rep], vocabularies[rep]
        (root / f'vocab_{rep}.json').write_text(
            json.dumps(vocabulary['token_to_index'], ensure_ascii=False),
            encoding='utf-8')
        for seed in seeds:
            run_id = f'{rep}_d45_he_s{seed}'
            model, final_state, best_state, history, rows, info = train_controlled_run(
                matrices, labels, configs[rep], rep, seed, 45, 'he', root / run_id)
            histories[run_id] = history
            gradients.extend(rows)
            infos.append(info)
            artifacts[(rep, seed, 45, 'he')] = info
            # Configuration déjà figée ; aucune adaptation ne dépend du test.
            for checkpoint, state in [('final', final_state), ('best_val', best_state)]:
                model.load_state_dict(state)
                for split in ('test_equilibre', 'test_pool'):
                    records.append({'model': 'DNN_' + rep, 'seed': int(seed),
                                    'split': split, 'checkpoint': checkpoint,
                                    'logits': _batched_logits(model, matrices[split]),
                                    'y_true': labels[split].astype(int),
                                    'message_ids': _message_ids(frames[split]),
                                    'n_parameters': info['n_parameters'],
                                    'fit_seconds': info['fit_seconds'],
                                    'run_id': run_id})
            del model, final_state, best_state
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print(f'Entraîné {run_id} : val BCE finale={info["final_val_bce"]:.4f}')
    # Un seul TF-IDF et une configuration fixée pour le plan factoriel.
    rep = ABLATION_REPRESENTATION
    ablation_rows = []
    for seed in seeds:
        for depth in ABLATION_DEPTHS:
            for init in ABLATION_INITIALIZATIONS:
                key = (rep, seed, depth, init)
                if key in artifacts:  # Réutiliser He/45 déjà calculé, sans réentraînement.
                    info = artifacts[key]
                else:
                    run_id = f'{rep}_d{depth}_{init}_s{seed}'
                    model, final_state, best_state, history, rows, info = train_controlled_run(
                        matrices_by_rep[rep], labels, configs[rep], rep, seed,
                        depth, init, root / run_id)
                    histories[run_id] = history
                    gradients.extend(rows)
                    infos.append(info)
                    del model, final_state, best_state
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    print(f'Contrôle entraîné {run_id}')
                ablation_rows.append(info)
    # Deux références classiques, mêmes exemples et TF-IDF que le DNN.
    # Même petite grille de régularisation pour les deux baselines ; seule
    # la validation intervient, et aucun scaler n'est fit sur validation/test.
    manual_matrices = {split: manual_features(frame['message'])
                       for split, frame in frames.items()}
    baseline_costs, baseline_selection = [], []
    c_grid = BASELINE_C_GRID
    for seed in seeds:
        for name, matrix in [('LR_tfidf', matrices_by_rep['tfidf']),
                             ('LR_manuel', manual_matrices)]:
            started = perf_counter()
            selected_estimator, selected_c, selected_f1 = None, None, -np.inf
            for c_value in c_grid:
                logistic = LogisticRegression(C=c_value, solver='lbfgs',
                    max_iter=3000, random_state=int(seed))
                candidate = (make_pipeline(StandardScaler(), logistic)
                             if name == 'LR_manuel' else logistic)
                candidate.fit(matrix['train'], labels['train'].astype(int))
                val_logits = candidate.decision_function(matrix['val'])
                val_f1 = float(f1_score(labels['val'].astype(int),
                                        val_logits >= 0., zero_division=0))
                baseline_selection.append({'model': name, 'seed': int(seed),
                                           'C': c_value, 'validation_f1': val_f1})
                # La grille est croissante : une égalité garde le plus petit C.
                if val_f1 > selected_f1:
                    selected_estimator = candidate
                    selected_c, selected_f1 = c_value, val_f1
            fit_seconds = perf_counter() - started
            estimator = selected_estimator
            logistic = estimator[-1] if name == 'LR_manuel' else estimator
            n_parameters = int(logistic.coef_.size + logistic.intercept_.size)
            baseline_costs.append({'model': name, 'seed': int(seed),
                                   'n_parameters': n_parameters,
                                   'fit_seconds': fit_seconds, 'selected_C': selected_c,
                                   'validation_f1': selected_f1})
            for split in ('test_equilibre', 'test_pool'):
                # decision_function fournit un logit sans clipping de probabilité.
                records.append({'model': name, 'seed': int(seed), 'split': split,
                                'checkpoint': 'final',
                                'logits': estimator.decision_function(matrix[split]),
                                'y_true': labels[split].astype(int),
                                'message_ids': _message_ids(frames[split]),
                                'n_parameters': n_parameters, 'fit_seconds': fit_seconds,
                                'selected_C': selected_c,
                                'run_id': f'{name}_s{seed}'})
    # Stockage portable des prédictions pour audit ; aucun chargement implicite.
    arrays, manifest = {}, []
    for i, record in enumerate(records):
        entry = {k: v for k, v in record.items()
                 if k not in ('logits', 'y_true', 'message_ids')}
        entry['array_key'] = str(i)
        manifest.append(entry)
        for key in ('logits', 'y_true', 'message_ids'):
            arrays[f'{i}_{key}'] = record[key]
    np.savez_compressed(root / 'predictions.npz', **arrays)
    (root / 'prediction_manifest.json').write_text(
        json.dumps(manifest, indent=2), encoding='utf-8')
    pd.DataFrame(infos).to_csv(root / 'training_summary.csv', index=False)
    pd.DataFrame(ablation_rows).to_csv(root / 'ablation_summary.csv', index=False)
    pd.DataFrame(baseline_costs).to_csv(root / 'baseline_costs.csv', index=False)
    pd.DataFrame(baseline_selection).to_csv(root / 'baseline_validation_C.csv', index=False)
    print('Résultats :', root.resolve())
    return records, histories, gradients, pd.DataFrame(ablation_rows), root


# Le notebook livre un protocole complet ; smoke=True est réservé aux tests.
experiment_records, training_histories, gradient_records, ablation_summary, FINAL_DIR = \
    run_experiments(smoke=False)
ablation_records = ablation_summary.to_dict('records')
display(ablation_summary)


## 7 — Convergence, propagation des gradients et surapprentissage

### Lire les gradients au bon moment

Les profils ci-dessous sont mesurés sur **le même mini-lot sonde, sans mise à jour des paramètres pendant la mesure** : à l'initialisation (`epoch=0`), avant le premier pas (`epoch=1`, donc le même état initial), puis avant les époques indiquées (`epoch=2` signifie qu'une époque a été effectuée). La sonde peut remettre à zéro les gradients, mais n'effectue aucun `optimizer.step()`. Les sondes 0 et 1 documentent le même état et ne constituent pas deux répétitions. L'époque 1 de ce nouveau protocole ne doit pas être confondue avec une moyenne de gradients sur toute la première époque dans l'ancien notebook.

Pour chaque couche, le code rapporte la norme de Frobenius et le gradient RMS :

\[
\operatorname{RMS}(\nabla W)=\frac{\|\nabla W\|_F}{\sqrt{\#W}}.
\]

La couche de sortie apparaît dans les données enregistrées, mais **le ratio premier/dernier utilise les couches cachées**. Le RMS facilite la comparaison de matrices de tailles différentes ; la norme brute dépend aussi du nombre de poids. Un ratio proche de 1 entre deux extrémités ne garantit pas que les couches intermédiaires transmettent correctement le signal. Il faut lire les profils complets, leur dispersion sur les graines et les contrôles 3/45 couches × quatre initialisations.

Dans les anciens logs, le ratio de normes arrondies était \(0{,}147/0{,}159\simeq0{,}925\). Ces deux normes étaient du même ordre au début de **cet ancien entraînement**. Ce constat, à lui seul, ne prouve pas que He « sauve » le réseau. Les anciennes expériences utilisaient un TF-IDF en comptages bruts ; leurs valeurs ne sont pas des résultats des expériences corrigées. Des gradients de l'ordre de \(10^{-8}\) lorsque la BCE d'entraînement est proche de \(10^{-10}\) sont compatibles avec une perte déjà minimisée et ne suffisent pas à diagnostiquer un vanishing gradient.

### Examiner le contrôle croisé

L'ablation fixe la représentation, le vocabulaire, la largeur, Adam, le taux d'apprentissage, la durée, les mini-lots et les graines. Seules la profondeur et l'initialisation des poids varient ; tous les biais sont initialisés à zéro puis appris normalement. Le contrôle « PyTorch » conserve les poids produits par `nn.Linear` et met ses biais à zéro : il ne correspond donc pas à l'ensemble des paramètres par défaut, biais compris. Le réseau de trois couches sert de contrôle ; le modèle principal conserve 45 couches cachées et plus de 1 000 époques.

On compare la propagation initiale, les pertes et les échecs de convergence. Une configuration donnant un gradient nul ou une perte non finie doit rester visible dans le bilan. Un avantage observé sur quelques graines et sur un seul découpage ne permet pas une conclusion universelle sur l'initialisation. Le coût en paramètres et en temps est rapporté à côté des performances ; les temps dépendent du matériel.

### Surapprentissage et confiance des prédictions

Les anciennes expériences avaient environ 1,7 million de paramètres pour 1 044 exemples d'apprentissage. Elles atteignaient une BCE d'entraînement presque nulle, alors que leur BCE de test se situait entre 1,11 et 1,96. Une prédiction constante \(p=0{,}5\) a une BCE de \(\ln2\simeq0{,}693\), quelle que soit la prévalence : **selon la BCE**, les anciens modèles étaient moins bons que cette référence, malgré leur bonne accuracy. Quelques erreurs extrêmement confiantes peuvent expliquer ce contraste. Une accuracy élevée ne garantit ni de bonnes probabilités ni une bonne calibration.

Le nouveau protocole réserve une validation interne ; son nombre d'exemples d'apprentissage et son rapport paramètres/exemples sont donc recalculés, sans réutiliser le dénominateur 1 044. Les courbes comparent la BCE d'apprentissage et de validation, et le tableau distingue le checkpoint final imposé par le sujet du checkpoint de BCE de validation minimale. L'entraînement principal se poursuit jusqu'au nombre d'époques prévu ; le checkpoint de validation est un diagnostic complémentaire. Sa sélection et le choix des hyperparamètres utilisent la même validation, ce qui limite l'interprétation de son niveau de performance.

Plus de 1 000 époques peuvent favoriser la surconfiance, mais leur rôle exact ne découle pas de la durée seule. La comparaison des checkpoints sur les mêmes graines permet d'étudier cette hypothèse. Les métriques de test, le score de Brier et les courbes de fiabilité sont rapportés en partie 8 ; on ne réajuste pas les modèles à partir de ces résultats. Le meilleur checkpoint de validation n'est pas automatiquement déclaré mieux calibré sans ces mesures.


In [ ]:
# PARTIE 7 — CONVERGENCE, GRADIENTS ET SURAPPRENTISSAGE
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def _convergence_frame(data, history=False):
    """Copier les tableaux ; accepter aussi un dictionnaire d'historiques."""
    if isinstance(data, pd.DataFrame):
        return data.copy()
    if isinstance(data, dict):
        rows = []
        for run_id, values in data.items():
            if isinstance(values, dict) and "history" in values:
                meta = {k: v for k, v in values.items() if k != "history"}
                entries = values["history"]
            else:
                meta, entries = {}, values
            if isinstance(entries, pd.DataFrame):
                entries = entries.to_dict("records")
            if isinstance(entries, dict):
                entries = [entries]
            rows.extend({"run_id": run_id, **meta, **row} for row in entries)
        return pd.DataFrame(rows)
    return pd.DataFrame(data).copy()


def analyser_convergence(training_histories, gradient_records, ablation_summary,
                        output_dir=None, n_train=None, training_summary=None):
    """Décrire les apprentissages ; aucun réglage à partir du test."""
    histories = _convergence_frame(training_histories, history=True)
    gradients = _convergence_frame(gradient_records)
    runs = _convergence_frame(training_summary if training_summary is not None else ablation_summary)
    # Les alias facilitent la lecture d'exports antérieurs du protocole corrigé.
    aliases = {"depth": "hidden_layers", "initialization": "init",
               "train_bce": "train_loss", "val_bce": "val_loss",
               "fro_norm": "grad_norm", "fro": "grad_norm", "rms": "grad_rms"}
    histories = histories.rename(columns={k: v for k, v in aliases.items()
                                         if k in histories and v not in histories})
    gradients = gradients.rename(columns={k: v for k, v in aliases.items()
                                         if k in gradients and v not in gradients})
    runs = runs.rename(columns={k: v for k, v in aliases.items()
                               if k in runs and v not in runs})
    expected_h = {"run_id", "epoch", "train_loss", "val_loss"}
    expected_g = {"run_id", "epoch", "layer_index", "grad_norm", "grad_rms"}
    if histories.empty or not expected_h.issubset(histories):
        raise ValueError(f"Historiques incomplets : colonnes attendues {expected_h}.")
    if gradients.empty or not expected_g.issubset(gradients):
        raise ValueError(f"Mesures de gradients incomplètes : {expected_g}.")
    meta_fields = ["representation", "hidden_layers", "init", "seed"]
    if not {"run_id", *meta_fields}.issubset(runs):
        raise ValueError("Le bilan des entraînements doit identifier représentation, profondeur, init et graine.")
    metadata = pd.concat([runs[["run_id", *meta_fields]],
                          gradients[["run_id", *meta_fields]]], ignore_index=True).drop_duplicates("run_id")
    for target in (histories, gradients):
        for key in meta_fields:
            if key not in target:
                target[key] = target["run_id"].map(metadata.set_index("run_id")[key])
        if target[meta_fields].isna().any().any():
            raise ValueError("Métadonnées manquantes pour certains entraînements.")
    histories = histories.sort_values(["run_id", "epoch"])
    gradients = gradients.sort_values(["run_id", "epoch", "layer_index"])
    if (gradients[["grad_norm", "grad_rms"]] < 0).any().any():
        raise ValueError("Une norme de gradient ne peut pas être négative.")

    # Aucune couche de sortie dans les ratios d'extrémités.
    if "layer_type" in gradients:
        hidden = gradients.loc[gradients["layer_type"].eq("hidden")].copy()
    elif "is_output" in gradients:
        hidden = gradients.loc[~gradients["is_output"].astype(bool)].copy()
    else:
        hidden = gradients.loc[gradients["layer_index"] < gradients["hidden_layers"]].copy()
    ratio_rows = []
    for (run_id, epoch), frame in hidden.groupby(["run_id", "epoch"], sort=False):
        frame = frame.sort_values("layer_index")
        first, last = frame.iloc[0], frame.iloc[-1]
        row = {"run_id": run_id, "epoch": int(epoch),
               **{field: first[field] for field in meta_fields},
               "n_hidden_measured": len(frame),
               "probe_loss": float(first.get("probe_loss", np.nan))}
        for name, column in [("fro", "grad_norm"), ("rms", "grad_rms")]:
            numerator, denominator = float(first[column]), float(last[column])
            row[f"first_{name}"] = numerator
            row[f"last_hidden_{name}"] = denominator
            row[f"ratio_{name}"] = numerator / denominator if denominator > 0 else np.nan
            row[f"ratio_{name}_status"] = (
                "défini" if denominator > 0 else "0/0" if numerator == 0 else "positif/0")
        row["zero_hidden_gradients"] = int(frame["grad_norm"].eq(0).sum())
        row["nonfinite_hidden_gradients"] = int(
            (~np.isfinite(frame[["grad_norm", "grad_rms"]].to_numpy())).any(axis=1).sum())
        ratio_rows.append(row)
    ratios = pd.DataFrame(ratio_rows)
    if ratios.empty:
        raise ValueError("Aucune couche cachée enregistrée.")
    early = ratios.loc[ratios["epoch"].isin([0, 1])]
    ratios_summary = early.groupby(
        ["representation", "hidden_layers", "init", "epoch"], dropna=False
    ).agg(n_runs=("seed", "nunique"),
          probe_loss_mean=("probe_loss", "mean"),
          fro_mean=("ratio_fro", "mean"), fro_std=("ratio_fro", "std"),
          fro_min=("ratio_fro", "min"), fro_max=("ratio_fro", "max"),
          rms_mean=("ratio_rms", "mean"), rms_std=("ratio_rms", "std"),
          rms_min=("ratio_rms", "min"), rms_max=("ratio_rms", "max"),
          zero_hidden_gradients=("zero_hidden_gradients", "sum"),
          nonfinite_hidden_gradients=("nonfinite_hidden_gradients", "sum")
          ).reset_index()
    print("Ratios premier/dernier caché : sonde initiale (0) et première rétropropagation (1), avant la première mise à jour.")
    print("Les sondes 0 et 1 sont volontairement identiques ; elles ne sont pas deux expériences indépendantes.")
    print("Un ratio indéfini (0/0 ou positif/0) reste NaN ; consulter les normes et les comptes de zéros.")
    display(ratios_summary.round(5))

    # Résumer les mêmes trajectoires : aucun arrêt anticipé du modèle principal.
    trajectory_rows = []
    for run_id, frame in histories.groupby("run_id", sort=False):
        finite_val = frame.loc[np.isfinite(frame["val_loss"])]
        final = frame.iloc[-1]
        if len(finite_val):
            best = finite_val.loc[finite_val["val_loss"].idxmin()]
            best_epoch, best_val = int(best["epoch"]), float(best["val_loss"])
        else:
            best_epoch, best_val = np.nan, np.nan
        trajectory_rows.append({"run_id": run_id,
            **{field: final[field] for field in meta_fields},
            "last_epoch": int(final["epoch"]), "best_val_epoch": best_epoch,
            "best_val_loss": best_val, "final_train_loss": float(final["train_loss"]),
            "final_val_loss": float(final["val_loss"]),
            "val_degradation": float(final["val_loss"]) - best_val,
            "final_val_minus_train": float(final["val_loss"] - final["train_loss"])})
    checkpoints = pd.DataFrame(trajectory_rows)
    extra = [column for column in ["n_parameters", "fit_seconds", "status", "error"]
             if column in runs]
    checkpoints = checkpoints.merge(runs[["run_id", *extra]].drop_duplicates("run_id"),
                                    on="run_id", how="left", validate="one_to_one")
    if n_train is not None:
        if int(n_train) <= 0:
            raise ValueError("n_train doit être strictement positif.")
        if "n_parameters" in checkpoints:
            checkpoints["parameters_per_train_example"] = checkpoints["n_parameters"] / int(n_train)
    print("BCE de validation minimale et checkpoint final, sur les mêmes apprentissages :")
    display(checkpoints.round(6))
    failure_mask = ~np.isfinite(checkpoints[["final_train_loss", "final_val_loss"]]).all(axis=1)
    if "status" in checkpoints:
        failure_mask |= ~checkpoints["status"].isin(["ok", "completed", "success", "cached"])
    failed = checkpoints.loc[failure_mask].copy()
    if not failed.empty:
        print("Entraînements échoués ou non finis — ils ne doivent pas être exclus silencieusement :")
        display(failed)

    directory = Path(output_dir) if output_dir is not None else None
    if directory is not None:
        directory.mkdir(parents=True, exist_ok=True)
    figure_paths = []
    def finish_figure(fig, stem):
        fig.tight_layout()
        if directory is not None:
            path = directory / f"{stem}.png"
            fig.savefig(path, dpi=160, bbox_inches="tight")
            figure_paths.append(str(path))
        plt.show()
        plt.close(fig)

    # Profils complets : chaque graine reste une observation d'apprentissage.
    # Le ruban indique min–max entre graines, et n'est PAS un IC d'échantillonnage.
    ablation_representation = "tfidf" if hidden["representation"].eq("tfidf").any() else hidden["representation"].iloc[0]
    for epoch in [0, 1, 2]:
        sample = hidden.loc[hidden["epoch"].eq(epoch)
                            & hidden["representation"].eq(ablation_representation)]
        if sample.empty:
            continue
        depths = sorted(sample["hidden_layers"].unique())
        available_inits = list(sample["init"].unique())
        init_order = [name for name in ["he", "xavier", "pytorch", "normal_001"] if name in available_inits]
        init_order += [name for name in available_inits if name not in init_order]
        fig, axes = plt.subplots(len(depths), len(init_order), figsize=(4 * len(init_order), 3.1 * len(depths)), squeeze=False, sharey=True)
        positives = sample[["grad_norm", "grad_rms"]].to_numpy()
        positives = positives[np.isfinite(positives) & (positives > 0)]
        floor = max(float(positives.min()) / 10, 1e-300) if len(positives) else 1e-30
        for i, depth in enumerate(depths):
            for j, init in enumerate(init_order):
                ax = axes[i, j]
                frame = sample.loc[sample["hidden_layers"].eq(depth) & sample["init"].eq(init)]
                if frame.empty:
                    ax.set_visible(False)
                    continue
                for column, label, color in [("grad_norm", "Frobenius", "#2563EB"), ("grad_rms", "RMS", "#D55E00")]:
                    profile = frame.groupby("layer_index")[column].agg(["median", "min", "max"])
                    x = profile.index.to_numpy(dtype=float) + 1
                    ax.plot(x, np.maximum(profile["median"].to_numpy(), floor), color=color, label=label)
                    ax.fill_between(x, np.maximum(profile["min"].to_numpy(), floor),
                                    np.maximum(profile["max"].to_numpy(), floor), color=color, alpha=.15)
                    zero_profile = profile["median"].eq(0).to_numpy()
                    if zero_profile.any():
                        ax.scatter(x[zero_profile], np.full(zero_profile.sum(), floor), color=color, marker="x")
                ax.set_yscale("log")
                ax.set_title(f"{depth} couches — {init}")
                ax.set_xlabel("Indice de couche cachée (1 à profondeur)")
                ax.set_ylabel("Norme ; médiane et min–max des graines")
                ax.grid(alpha=.2)
                ax.legend(fontsize=8)
                zero_count = int(frame["grad_norm"].eq(0).sum())
                if zero_count:
                    ax.text(.02, .03, f"{zero_count} normes nulles ; × au plancher d'affichage", transform=ax.transAxes, fontsize=7)
        probe_label = {0: "initiale, avant toute mise à jour",
                       1: "avant le premier pas (même état que 0)",
                       2: "avant la deuxième époque"}[epoch]
        fig.suptitle(f"{ablation_representation} — sonde {probe_label}", y=1.02)
        finish_figure(fig, f"gradients_complets_epoque_{epoch}")

    # Toutes les couches/représentations He45 restent dans les tableaux de ratios.
    main_histories = histories.loc[histories["hidden_layers"].eq(45) & histories["init"].eq("he")]
    if main_histories.empty:
        main_histories = histories
    representations = list(main_histories["representation"].unique())
    fig, axes = plt.subplots(1, len(representations), figsize=(5 * len(representations), 4), squeeze=False)
    for j, representation in enumerate(representations):
        ax = axes[0, j]
        sample = main_histories.loc[main_histories["representation"].eq(representation)]
        for column, label, color in [("train_loss", "Apprentissage", "#2563EB"), ("val_loss", "Validation", "#D55E00")]:
            summary = sample.groupby("epoch")[column].agg(["median", "min", "max"])
            x = summary.index.to_numpy()
            ax.plot(x, np.maximum(summary["median"].to_numpy(), 1e-15), color=color, label=label)
            ax.fill_between(x, np.maximum(summary["min"].to_numpy(), 1e-15), np.maximum(summary["max"].to_numpy(), 1e-15), color=color, alpha=.15)
        ax.axhline(np.log(2), color="black", linestyle=":", label="p=0,5 : ln 2")
        ax.set(title=f"{representation} — He, 45 couches", xlabel="Époque", ylabel="BCE (échelle logarithmique)")
        ax.set_yscale("log")
        ax.legend(fontsize=8)
        ax.grid(alpha=.2)
    finish_figure(fig, "pertes_apprentissage_validation")

    # Contrôle croisé : mêmes hyperparamètres et même représentation.
    controlled = histories.loc[histories["representation"].eq(ablation_representation)]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for (depth, init), frame in controlled.groupby(["hidden_layers", "init"]):
        for ax, column in zip(axes, ["train_loss", "val_loss"]):
            profile = frame.groupby("epoch")[column].median()
            ax.plot(profile.index, np.maximum(profile.to_numpy(), 1e-15), label=f"{depth} — {init}")
    for ax, title in zip(axes, ["Apprentissage", "Validation"]):
        ax.axhline(np.log(2), color="black", linestyle=":")
        ax.set(title=f"{ablation_representation} — {title}", xlabel="Époque", ylabel="BCE médiane sur graines")
        ax.set_yscale("log")
        ax.legend(fontsize=7, ncol=2)
        ax.grid(alpha=.2)
    finish_figure(fig, "ablation_pertes")
    print("Les rubans des courbes représentent la dispersion min–max entre graines, pas des IC sur les messages.")
    print("Les valeurs nulles sont conservées dans les tableaux ; un plancher n'est utilisé que pour l'affichage logarithmique.")
    print("Des gradients faibles après une BCE presque nulle ne prouvent pas un défaut de propagation.")
    outputs = {"gradient_ratios": ratios, "ratios_summary": ratios_summary,
               "checkpoints": checkpoints, "failed_runs": failed,
               "histories": histories, "gradient_profiles": gradients,
               "figure_paths": figure_paths}
    if directory is not None:
        for name in ["gradient_ratios", "ratios_summary", "checkpoints", "failed_runs"]:
            outputs[name].to_csv(directory / f"{name}.csv", index=False)
    return outputs


# Exécuter après la partie 6 : aucun entraînement supplémentaire.
convergence_results = analyser_convergence(
    training_histories, gradient_records, ablation_summary,
    output_dir=Path(FINAL_DIR) / "convergence",
    n_train=len(train_df),
    training_summary=pd.read_csv(Path(FINAL_DIR) / "training_summary.csv"),
)


## 8 — Évaluation, incertitude et prévalence

Le seuil de décision est fixé à **0,5 avant l’évaluation**. Chaque modèle est évalué sur les mêmes identifiants de messages, pour chaque graine, sur le test équilibré et sur `test_pool` conservé à sa prévalence observée. Ces deux tests se recouvrent : ils ne constituent pas deux réplications indépendantes. Les poids, vocabulaire, IDF, paramètres de normalisation et hyperparamètres restent figés.

Les tableaux rapportent accuracy, précision, rappel, F1, FPR, **ROC-AUC**, **précision moyenne** (`average_precision_score`), **BCE calculée à partir des logits** et **score de Brier**. La précision moyenne est la somme des gains de rappel pondérés par la précision, et diffère de l’aire précision–rappel obtenue par interpolation trapézoïdale. ROC-AUC et AP utilisent les scores continus sans imposer un seuil ; **la courbe précision–rappel et AP dépendent de la prévalence**. Les graphiques illustrent la plus petite graine, désignée à l’avance ; les tableaux incluent toutes les graines.

Les IC à 95 % rééchantillonnent séparément **les graines et les messages**. Un même tirage de messages est partagé par toutes les graines et, pour une différence A − B, par les deux modèles. Le bootstrap est stratifié sur le test équilibré ; sur `test_pool`, le tirage des messages permet aussi à la prévalence de varier. Cinq graines sur 262 messages ne forment donc jamais 1 310 observations indépendantes. Ces IC restent conditionnels au découpage des données et à la configuration choisie : ils ne couvrent ni un nouveau jeu de données ni l’incertitude de la recherche Optuna. Avec seulement cinq graines, l’estimation de la variabilité d’entraînement reste limitée. Les IC des différences sont exploratoires et ne sont pas des IC simultanés.

Le **McNemar exact est calculé par graine**, sur les discordances de correction pour l’accuracy, puis uniquement sur les ham pour comparer les faux positifs. Il ne compare pas directement les F1. La correction de Holm porte sur la famille complète des tests affichés, comprenant paires, graines, critères et splits ; aucune p-value n’est obtenue en concaténant les prédictions des graines. Les tests sur des splits qui se recouvrent sont dépendants ; Holm n’exige pas leur indépendance. Un résultat non significatif ne prouve pas l’équivalence. Dans les anciens logs, BoW et TF-IDF avaient chacun 14 erreurs : leurs discordances globales sont nécessairement égales et McNemar exact aurait donné $p=1$, sans prouver leur équivalence. Ces anciens chiffres ne remplacent pas les nouvelles expériences.

La projection de précision à une prévalence $\pi$ suppose que le rappel et le taux de faux positifs restent inchangés :

$$\mathrm{Précision}(\pi)=\frac{\pi\,\mathrm{Rappel}}{\pi\,\mathrm{Rappel}+(1-\pi)\,\mathrm{FPR}}.$$

À **13 %**, les anciennes matrices donnaient environ **57,2 % (binaire), 81,9 % (BoW), 89,9 % (TF-IDF)**. Ce sont des projections historiques et conditionnelles ; 13 % n’est pas une prévalence universelle du spam. Le code recalcule cette projection pour chaque nouvelle graine et rapporte séparément les mesures directes sur `test_pool`.

Une BCE de test supérieure à $\ln 2\simeq0,693$ est moins bonne que la prédiction constante $p=0,5$ **selon la BCE**, même avec une bonne accuracy. Le score de Brier et les courbes de fiabilité quantifient ou montrent la qualité des probabilités. Les erreurs très confiantes doivent être discutées. Le checkpoint de meilleure BCE de validation est évalué à côté du checkpoint final imposé, sans choisir le meilleur après lecture du test ; leur comparaison aide à étudier le rôle de la durée d’apprentissage.


In [ ]:
# PARTIE 8 — Évaluation appariée, incertitude, calibration et prévalence
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit
from scipy.stats import binomtest
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score
from IPython.display import display

METRIC_NAMES = (
    'accuracy', 'precision', 'recall', 'f1', 'fpr',
    'bce', 'brier', 'roc_auc', 'average_precision',
)
COMPARISON_PAIRS = (
    ('DNN_binaire', 'DNN_bow'), ('DNN_binaire', 'DNN_tfidf'),
    ('DNN_bow', 'DNN_tfidf'), ('DNN_tfidf', 'LR_tfidf'),
    ('DNN_tfidf', 'LR_manuel'),
)


def metric_vector(y, logits, weights=None):
    """Métriques au seuil logit 0 ; BCE stable, AUC/AP avec gestion des ex æquo.

    Les poids ne servent qu'au bootstrap des mêmes messages ; ils ne changent
    ni le seuil, ni l'apprentissage. AP = somme (rappel_i-rappel_{i-1}) précision_i.
    """
    y = np.asarray(y, dtype=np.int64)
    z = np.asarray(logits, dtype=np.float64)
    w = np.ones(len(y), dtype=np.float64) if weights is None else np.asarray(weights, dtype=np.float64)
    if y.ndim != 1 or z.shape != y.shape or w.shape != y.shape:
        raise ValueError('Labels, logits et poids doivent avoir la même forme 1D.')
    if not np.isin(y, [0, 1]).all() or not np.isfinite(z).all():
        raise ValueError('Labels binaires et logits finis requis.')
    if not np.isfinite(w).all() or (w < 0).any() or w.sum() <= 0:
        raise ValueError('Poids finis, positifs ou nuls, de somme non nulle requis.')
    pred = z >= 0
    tp = np.sum(w * (y == 1) * pred)
    tn = np.sum(w * (y == 0) * ~pred)
    fp = np.sum(w * (y == 0) * pred)
    fn = np.sum(w * (y == 1) * ~pred)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else np.nan
    fpr = fp / (fp + tn) if fp + tn else np.nan
    f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0
    bce = np.average(np.logaddexp(0.0, z) - y * z, weights=w)
    brier = np.average((expit(z) - y) ** 2, weights=w)
    positives = np.sum(w * y)
    negatives = np.sum(w * (1 - y))
    if positives and negatives:
        order = np.argsort(-z, kind='stable')
        threshold_ends = np.r_[np.flatnonzero(np.diff(z[order])), len(y) - 1]
        tps = np.cumsum(w[order] * y[order])[threshold_ends]
        fps = np.cumsum(w[order] * (1 - y[order]))[threshold_ends]
        recalls = tps / positives
        precisions = np.divide(tps, tps + fps, out=np.zeros_like(tps), where=(tps + fps) > 0)
        average_precision = np.sum(np.diff(np.r_[0.0, recalls]) * precisions)
        # Formule trapézoïdale pour ROC uniquement ; pas pour la précision moyenne.
        rx, ry = np.r_[0.0, fps / negatives], np.r_[0.0, recalls]
        roc_auc = np.sum(np.diff(rx) * (ry[1:] + ry[:-1]) / 2)
    else:
        roc_auc, average_precision = np.nan, np.nan
    return np.array([
        (tp + tn) / w.sum(), precision, recall, f1, fpr,
        bce, brier, roc_auc, average_precision,
    ], dtype=np.float64)


def _normalize_record(record):
    r = dict(record)
    y = np.asarray(r['y_true'], dtype=np.int64)
    ids = np.asarray(r['message_ids']).astype(str)
    if 'logits' in r and r['logits'] is not None:
        z = np.asarray(r['logits'], dtype=np.float64)
    else:
        p = np.asarray(r['probabilities'], dtype=np.float64)
        if not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
            raise ValueError('Probabilités hors de [0, 1].')
        p = np.clip(p, np.finfo(float).eps, 1 - np.finfo(float).eps)
        z = np.log(p) - np.log1p(-p)
    if ids.shape != y.shape or z.shape != y.shape or len(set(ids)) != len(ids):
        raise ValueError('Identifiants uniques, labels et logits alignés requis.')
    if len(np.unique(y)) != 2:
        raise ValueError('Chaque split doit contenir les deux classes.')
    metric_vector(y, z)  # validation commune
    order = np.argsort(ids, kind='stable')
    r.update(y_true=y[order], logits=z[order], message_ids=ids[order])
    return r


def _assert_aligned(records):
    ref = records[0]
    for r in records[1:]:
        if not np.array_equal(ref['message_ids'], r['message_ids']) or not np.array_equal(ref['y_true'], r['y_true']):
            raise ValueError('Un même split doit garder exactement les mêmes messages et labels.')


def _bootstrap_weights(y, rng, stratified):
    """Une seule sélection de messages, partagée par tous les modèles/graines."""
    n = len(y)
    if stratified:
        groups = [np.flatnonzero(y == label) for label in (0, 1)]
        sample = np.concatenate([rng.choice(g, len(g), replace=True) for g in groups])
    else:
        sample = rng.integers(0, n, size=n)
    return np.bincount(sample, minlength=n)


def bootstrap_runs(records_a, records_b=None, *, n_bootstrap=1000, seed=2026,
                   stratified=True):
    """IC à deux facteurs : graines et messages rééchantillonnés séparément.

    Pour A-B, on réutilise les mêmes graines ET les mêmes messages. Une graine
    n'ajoute aucun message indépendant. Les IC ne couvrent pas l'incertitude
    due au découpage train/test, ni celle de la recherche d'hyperparamètres.
    """
    if n_bootstrap < 200:
        raise ValueError('Au moins 200 rééchantillonnages pour les IC (1000+ conseillés).')
    a = {int(r['seed']): r for r in records_a}
    b = None if records_b is None else {int(r['seed']): r for r in records_b}
    seeds = sorted(a if b is None else set(a) & set(b))
    if not seeds:
        raise ValueError('Aucune graine commune pour la comparaison appariée.')
    records = [a[s] for s in seeds] + ([] if b is None else [b[s] for s in seeds])
    _assert_aligned(records)
    y = a[seeds[0]]['y_true']
    rng = np.random.default_rng(seed)
    samples = np.empty((n_bootstrap, len(METRIC_NAMES)))
    for draw in range(n_bootstrap):
        w = _bootstrap_weights(y, rng, stratified)
        # Multinomial des graines ; on calcule chaque graine une seule fois.
        seed_counts = rng.multinomial(len(seeds), np.full(len(seeds), 1 / len(seeds)))
        value = np.zeros(len(METRIC_NAMES))
        for count, s in zip(seed_counts, seeds):
            if count:
                delta = metric_vector(y, a[s]['logits'], w)
                if b is not None:
                    delta -= metric_vector(y, b[s]['logits'], w)
                value += count * delta / len(seeds)
        samples[draw] = value
    point = np.mean([metric_vector(y, a[s]['logits']) - (
        metric_vector(y, b[s]['logits']) if b is not None else 0.0
    ) for s in seeds], axis=0)
    low, high = np.nanquantile(samples, [0.025, 0.975], axis=0)
    return pd.DataFrame({
        'metric': METRIC_NAMES, 'estimate': point, 'ci95_low': low,
        'ci95_high': high, 'n_seeds': len(seeds), 'n_messages': len(y),
    })


def _holm(pvalues):
    p = np.asarray(pvalues, dtype=float)
    order = np.argsort(p)
    adjusted = np.maximum.accumulate((len(p) - np.arange(len(p))) * p[order])
    result = np.empty(len(p))
    result[order] = np.minimum(adjusted, 1.0)
    return result


def mcnemar_per_seed(records, pairs=COMPARISON_PAIRS):
    """McNemar exact sur erreurs globales et FP parmi les seuls ham."""
    rows = []
    splits = sorted({r['split'] for r in records})
    for split in splits:
        lookup = {(r['model'], int(r['seed'])): r for r in records if r['split'] == split}
        for model_a, model_b in pairs:
            seeds = sorted({s for m, s in lookup if m == model_a} & {s for m, s in lookup if m == model_b})
            for seed in seeds:
                a, b = lookup[(model_a, seed)], lookup[(model_b, seed)]
                _assert_aligned([a, b])
                y = a['y_true']
                wrong_a, wrong_b = (a['logits'] >= 0) != y, (b['logits'] >= 0) != y
                for outcome, mask in [('accuracy', np.ones(len(y), dtype=bool)), ('FP_ham', y == 0)]:
                    # n10 : A seul se trompe ; n01 : B seul se trompe.
                    n10 = int(np.sum(mask & wrong_a & ~wrong_b))
                    n01 = int(np.sum(mask & ~wrong_a & wrong_b))
                    p = binomtest(n10, n10 + n01, p=0.5).pvalue if n10 + n01 else 1.0
                    rows.append(dict(split=split, model_a=model_a, model_b=model_b,
                                     seed=seed, outcome=outcome, n10=n10, n01=n01,
                                     n_messages=int(mask.sum()), p_exact=p))
    table = pd.DataFrame(rows)
    if not table.empty:
        # Une famille : toutes les comparaisons finales affichées (splits/graines/critères).
        table['p_holm'] = _holm(table['p_exact'])
    return table


def _plot_reference_runs(records, output_dir=None):
    """Illustrations pour la plus petite graine, fixée avant lecture du test."""
    figures = []
    for split in sorted({r['split'] for r in records}):
        selected = [r for r in records if r['split'] == split]
        fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
        for model in sorted({r['model'] for r in selected}):
            r = min((r for r in selected if r['model'] == model), key=lambda x: int(x['seed']))
            y, z = r['y_true'], r['logits']
            fpr, tpr, _ = roc_curve(y, z)
            precision, recall, _ = precision_recall_curve(y, z)
            fraction, mean_probability = calibration_curve(y, expit(z), n_bins=8, strategy='uniform')
            label = f"{model} (graine {r['seed']})"
            axes[0].plot(fpr, tpr, label=label)
            axes[1].plot(recall, precision, label=label)
            axes[2].plot(mean_probability, fraction, marker='o', label=label)
        axes[0].plot([0, 1], [0, 1], 'k--', alpha=.5)
        axes[1].axhline(selected[0]['y_true'].mean(), linestyle='--', color='grey', label='Prévalence observée')
        axes[2].plot([0, 1], [0, 1], 'k--', alpha=.5)
        for ax, title, xlabel, ylabel in zip(axes,
                ['ROC', 'Précision–rappel', 'Courbe de fiabilité'],
                ['FPR', 'Rappel', 'Probabilité moyenne prédite'],
                ['Rappel', 'Précision', 'Fréquence observée du spam']):
            ax.set(title=title, xlabel=xlabel, ylabel=ylabel, xlim=(0, 1), ylim=(0, 1.02))
            ax.grid(alpha=.2)
        axes[0].legend(fontsize=7, loc='lower right')
        fig.suptitle(f'{split} — checkpoint final, graine de référence (illustration)')
        fig.tight_layout()
        if output_dir is not None:
            fig.savefig(Path(output_dir) / f'courbes_{split}.png', dpi=160, bbox_inches='tight')
        plt.show()
        figures.append(fig)
    return figures


def evaluate_experiments(experiment_records, output_dir=None, n_bootstrap=1000,
                         make_plots=True):
    records = [_normalize_record(r) for r in experiment_records]
    keys = [(r['model'], int(r['seed']), r['split'], r['checkpoint']) for r in records]
    if len(set(keys)) != len(keys):
        raise ValueError('Chaque modèle/graine/split/checkpoint doit apparaître une seule fois.')
    final_records = [r for r in records if r['checkpoint'] == 'final']
    if not final_records:
        raise ValueError('Aucun checkpoint final à évaluer.')
    if output_dir is not None:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for r in records:
        row = {k: r[k] for k in ['model', 'seed', 'split', 'checkpoint']}
        row.update(zip(METRIC_NAMES, metric_vector(r['y_true'], r['logits'])))
        # Les métriques ponctuelles utilisent directement les API de référence.
        # Le calcul équivalent de metric_vector accélère le bootstrap pondéré.
        row['roc_auc'] = roc_auc_score(r['y_true'], r['logits'])
        row['average_precision'] = average_precision_score(r['y_true'], r['logits'])
        y, pred = r['y_true'], r['logits'] >= 0
        row.update(n_messages=len(y), prevalence=float(y.mean()),
                   tn=int(np.sum((y == 0) & ~pred)), fp=int(np.sum((y == 0) & pred)),
                   fn=int(np.sum((y == 1) & ~pred)), tp=int(np.sum((y == 1) & pred)),
                   n_parameters=r.get('n_parameters', np.nan),
                   fit_seconds=r.get('fit_seconds', np.nan))
        rows.append(row)
    per_run = pd.DataFrame(rows)
    finals = per_run[per_run.checkpoint == 'final'].copy()
    summary = finals.groupby(['split', 'model'])[list(METRIC_NAMES) + ['n_parameters', 'fit_seconds']].agg(['mean', 'std'])
    print('Métriques finales par graine ; seuil fixé à p=0,5, aucun réglage sur le test.')
    display(finals.round(5))
    print('Moyenne et écart-type entre graines (une seule graine donne un écart-type indéfini).')
    display(summary.round(5))
    ci_tables, paired_tables = [], []
    for split in sorted({r['split'] for r in final_records}):
        subset = [r for r in final_records if r['split'] == split]
        _assert_aligned(subset)
        models = sorted({r['model'] for r in subset})
        for model in models:
            ci = bootstrap_runs([r for r in subset if r['model'] == model],
                                n_bootstrap=n_bootstrap, stratified=(split == 'test_equilibre'))
            ci['split'], ci['model'] = split, model
            ci_tables.append(ci)
        for model_a, model_b in COMPARISON_PAIRS:
            if model_a in models and model_b in models:
                ci = bootstrap_runs([r for r in subset if r['model'] == model_a],
                                    [r for r in subset if r['model'] == model_b],
                                    n_bootstrap=n_bootstrap, stratified=(split == 'test_equilibre'))
                ci['split'], ci['model_a'], ci['model_b'] = split, model_a, model_b
                paired_tables.append(ci)
    intervals = pd.concat(ci_tables, ignore_index=True)
    paired_intervals = pd.concat(paired_tables, ignore_index=True) if paired_tables else pd.DataFrame()
    mcnemar = mcnemar_per_seed(final_records)
    print('IC 95 % bootstrap graines/messages ; différences A − B appariées.')
    display(intervals.round(5))
    display(paired_intervals.round(5))
    print('McNemar exact par graine : accuracy et FP parmi les ham ; correction de Holm globale.')
    display(mcnemar.round(5))
    projection = finals[finals['split'] == 'test_equilibre'][['model', 'seed', 'precision', 'recall', 'fpr']].copy()
    projection['prevalence_scenario'] = 0.13
    numerator = .13 * projection['recall']
    denominator = numerator + .87 * projection['fpr']
    projection['precision_at_13pct'] = np.divide(numerator, denominator,
        out=np.full(len(projection), np.nan), where=denominator.to_numpy() > 0)
    print('Projection à 13 % : hypothèse de rappel et FPR inchangés. Ce scénario ne remplace pas test_pool.')
    display(projection.round(5))
    # Comparaison contrôlée des checkpoints sur les mêmes exemples, sans choix par le test.
    checkpoints = per_run[per_run['checkpoint'].isin(['final', 'best_val'])].pivot(
        index=['split', 'model', 'seed'], columns='checkpoint', values=list(METRIC_NAMES))
    checkpoint_comparison = checkpoints.copy()
    if 'best_val' in checkpoints.columns.get_level_values(1):
        for metric in METRIC_NAMES:
            if (metric, 'final') in checkpoints and (metric, 'best_val') in checkpoints:
                checkpoint_comparison[(metric, 'final_minus_best_val')] = checkpoints[(metric, 'final')] - checkpoints[(metric, 'best_val')]
        print('Checkpoint choisi par BCE de validation et checkpoint final imposé ; comparaison descriptive.')
        display(checkpoint_comparison.round(5))
    print(f'Référence constante p=0,5 : BCE = ln(2) = {np.log(2):.6f}, Brier = 0,25.')
    print("Une BCE supérieure à ln(2) signifie une qualité probabiliste inférieure à cette référence ; elle ne signifie pas une accuracy inférieure à 50 %. Une absence de différence significative ne prouve pas l’équivalence.")
    tables = dict(per_run=per_run, summary=summary, intervals=intervals,
                  paired_intervals=paired_intervals, mcnemar=mcnemar,
                  checkpoint_comparison=checkpoint_comparison,
                  prevalence_projection=projection)
    if output_dir is not None:
        for name, table in tables.items():
            table.to_csv(output_dir / f'{name}.csv', index=(name in {'summary', 'checkpoint_comparison'}))
    if make_plots:
        tables['figures'] = _plot_reference_runs(final_records, output_dir)
    return tables


# Les tableaux reposent sur les nouvelles prédictions de la partie 6.
if 'experiment_records' in globals():
    evaluation_results = evaluate_experiments(
        experiment_records,
        output_dir=Path(globals().get('FINAL_DIR', globals().get('RESULTS_DIR', 'resultats_corriges'))) / 'evaluation',
        n_bootstrap=1000,
    )


## 9 — Synthèse, compromis et limites

La conclusion doit s'appuyer sur les **nouveaux tableaux**, et indiquer la dispersion entre graines, les intervalles appariés et le coût de calcul. Une égalité d'accuracy ou un test non significatif ne démontre pas l'équivalence. Les résultats obtenus après plusieurs graines sur les mêmes SMS ne multiplient pas le nombre de messages indépendants.

**Propagation des gradients.** Le ratio historique $0{,}147/0{,}159\simeq0{,}925$ indique des normes de même ordre entre les deux couches mesurées au début de cet entraînement. Il ne prouve pas, seul, que He sauve un réseau de 45 couches. Les profils complets, leur RMS et les contrôles croisés profondeur × initialisation permettent de préciser cette conclusion. Des gradients très faibles lorsque la BCE d'apprentissage est presque nulle ne démontrent pas un défaut de propagation.

**Généralisation et confiance.** Dans l'ancienne version, environ 1,7 million de paramètres pour 1 044 exemples, une BCE d'apprentissage presque nulle et une BCE de test de 1,11 à 1,96 signalaient un écart important. Cette BCE dépasse $\ln 2\simeq0{,}693$, référence d'une prédiction constante $p=0{,}5$, malgré une bonne accuracy. Il faut examiner les erreurs très confiantes, le score de Brier et la courbe de fiabilité. Les nouveaux effectifs, nombres de paramètres et résultats sont calculés dans cette version. Les 1 001 époques peuvent contribuer au surapprentissage ; la comparaison finale/best-validation donne un diagnostic sur les checkpoints, sans identifier à elle seule toutes les causes.

**Représentations et prévalence.** Les anciens BoW et TF-IDF comptaient chacun 14 erreurs : leur McNemar exact global donne $p=1$, ce qui ne prouve pas leur équivalence. Les différences de FP doivent être analysées parmi les ham. Sous l'hypothèse de rappel et FPR inchangés, les projections historiques à 13 % étaient d'environ 57,2 % (binaire), 81,9 % (BoW) et 89,9 % (TF-IDF) de précision. Ce sont des scénarios issus des anciens résultats, pas de nouvelles mesures. La nouvelle évaluation utilise directement `test_pool`, en plus du test équilibré. L'average precision et la courbe précision–rappel dépendent de la prévalence.

**DNN et références classiques.** Comparer le DNN TF-IDF à la régression logistique sur les mêmes vecteurs et à celle sur les caractéristiques manuelles : longueur, majuscules, chiffres, liens et monnaies. Les performances des baselines sont mesurées, sans présumer qu'elles égalent le réseau. Un éventuel bénéfice du DNN doit être mis en regard des paramètres, du temps d'ajustement, de la qualité probabiliste et du coût de sélection.

**Portée.** Le corpus SMS, le petit test équilibré, le découpage unique, la recherche bornée, la sélection sur validation et le faible nombre de graines limitent la généralisation des conclusions. Les intervalles exploratoires conditionnent sur l'apprentissage et les réglages retenus ; ils ne couvrent pas toutes ces sources d'incertitude. Aucun classement définitif ni garantie de performance en production ne doit être déduit de ces seules expériences.


In [ ]:
# Tableau compact pour rédiger la conclusion à partir des seules nouvelles mesures.
final_metrics = evaluation_results['per_run'].query("checkpoint == 'final'")
synthesis_table = final_metrics.groupby(['split', 'model']).agg(
    accuracy_moyenne=('accuracy', 'mean'), accuracy_ecart_type=('accuracy', 'std'),
    f1_moyen=('f1', 'mean'), bce_moyenne=('bce', 'mean'),
    brier_moyen=('brier', 'mean'), roc_auc_moyenne=('roc_auc', 'mean'),
    average_precision_moyenne=('average_precision', 'mean'),
    parametres=('n_parameters', 'first'), temps_ajustement_moyen_s=('fit_seconds', 'mean')
)
display(synthesis_table.round(5))
synthesis_table.to_csv(FINAL_DIR / 'synthese.csv')
print('Lire aussi les IC appariés, McNemar, les courbes de fiabilité et les contrôles de la partie 7.')
print('Le temps par modèle ne comprend pas la recherche Optuna ; les baselines incluent leur sélection de C.')
